# 12 — Multi-head Latent Attention (MLA)

**Prerequisite:** notebook **11** (why we skip “V1” and go GPT-2 → **V2**).

**Paper:** DeepSeek-V2 §2.1 — [arxiv](https://arxiv.org/html/2405.04434)

---

## GPT-2 MHA (notebook 4 + llm.c)

- One linear produces **Q, K, V** (width `3C`).
- At inference you cache **full** `K` and `V` for every head → KV memory grows fast with context length.

## MLA in one sentence

Compress keys+values into a **small latent** `c_kv` per token; **reconstruct** K/V when computing attention.

At inference you mainly store **`c_kv`** (dim = `kv_lora_rank`), not `2 × n_head × head_dim`.

## Learning goal

See `last_c_kv` on the module, compare cache bytes to GPT-2, then read **`c/deepseek_v2/mla.c`** (same math, more comments).


In [ ]:
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention

# Tiny config — same numbers as c/deepseek_v2/test_mla.c (C=128, 4 heads, kv_lora_rank=32)
cfg = DeepSeekV2Config.tiny(vocab_size=128, block_size=32)
x = torch.randn(2, 16, cfg.n_embd)  # batch=2, time=16, channels=128

attn = MultiHeadLatentAttention(cfg)

# Forward path (read deepseek_v2.py alongside):
#   q  = wq(x)
#   c_kv = w_dkv(x)          <-- THIS is what V2 wants in the KV cache
#   k  = w_uk(c_kv) per head
#   v  = w_uv(c_kv) per head
#   then causal softmax attention (same as notebook 4)
y = attn(x)

print("input:", tuple(x.shape))
print("output:", tuple(y.shape))
print("latent c_kv cached:", tuple(attn.last_c_kv.shape), "  # (B, T, kv_lora_rank)")
print("head_dim:", cfg.n_embd // cfg.n_head, "| kv_lora_rank:", cfg.kv_lora_rank)


In [ ]:
# Compare KV cache footprint (educational estimate, float32)
from llmc.deepseek_v2 import DeepSeekV2

model = DeepSeekV2(cfg)
print("MLA cache bytes/token (all layers):", model.kv_cache_bytes_per_token())
print("MHA cache bytes/token (hypothetical GPT-2):", model.mha_kv_cache_bytes_per_token())
print("ratio MHA/MLA:", model.mha_kv_cache_bytes_per_token() / model.kv_cache_bytes_per_token())


## Step-by-step (read the code)

1. `w_dkv(x)` → **`c_kv`** with shape `(B, T, kv_lora_rank)` — this is the **compressed cache**.
2. `w_uk(c_kv)` and `w_uv(c_kv)` → full keys and values per head (reconstructed when needed).
3. Causal softmax attention — **same math as notebook 4**, different way to build K/V.

## C port (piece 1 — done)

```bash
cd c && make test_mla && ./bin/test_mla
```

Open **`c/deepseek_v2/mla.c`** — comments map line-by-line to `MultiHeadLatentAttention` in `llmc/deepseek_v2.py`.

In **`vendor/llm.c/train_gpt2.c`**, search for `attention_forward`; MLA replaces that block once we wire weights into a full trainer.
